# Stage 04 - Apply Finding Rules

Apply versioned deterministic rules to streaming measures and preserve evidence references for every published finding.

In [ ]:
import json
from pyspark.sql import functions as F

measures_df = spark.table('silver_streaming_measures')
contract_issues_df = spark.table('silver_stream_contract_issues')
expectations_df = spark.table('silver_stream_expectations')

rules_path = 'Files/shared/integrated-test-data/projections/realtime/finding_rules.json'
rules_text = '\n'.join(row.value for row in spark.read.text(rules_path).collect())
rules = json.loads(rules_text)
rules_by_code = {rule['finding_code']: rule for rule in rules['finding_rules']}

ruleset_id = rules['ruleset_id']
ruleset_version = rules['ruleset_version']


In [ ]:
def apply_rule_metadata(frame, rule_code, title, threshold_text):
    rule = rules_by_code[rule_code]
    return (
        frame.withColumn('ruleset_id', F.lit(ruleset_id))
        .withColumn('ruleset_version', F.lit(ruleset_version))
        .withColumn('rule_id', F.lit(rule['rule_id']))
        .withColumn('rule_version', F.lit(rule['rule_version']))
        .withColumn('finding_code', F.lit(rule['finding_code']))
        .withColumn('finding_category', F.lit(rule['finding_category']))
        .withColumn('severity', F.lit(rule['severity']))
        .withColumn('finding_title', F.lit(title))
        .withColumn('threshold_text', F.lit(threshold_text))
        .withColumn('finding_status', F.lit('OPEN'))
        .withColumn('finding_version', F.lit(ruleset_version))
        .withColumn(
            'finding_id',
            F.concat(
                F.lit('finding-'),
                F.lower(F.substring(F.sha2(F.concat_ws('|', F.lit(rule['rule_id']), F.col('evidence_key')), 256), 1, 16)),
            ),
        )
    )


In [ ]:
duplicate_hits_base_df = contract_issues_df.filter(F.col('issue_type') == 'DUPLICATE_EVENT_ID').select(
    F.col('record_id').alias('evidence_key'),
    F.array(F.col('record_id')).alias('evidence_event_ids'),
    F.col('ordering_key').alias('affected_ordering_key'),
    F.col('ingest_time_ts').alias('finding_event_time_utc'),
    F.lit('duplicate_rank').alias('measure_name'),
    F.concat(F.lit('duplicate_rank='), F.col('duplicate_rank').cast('string')).alias('measured_value_text'),
    F.concat(F.lit('Replay preserved a repeated event_id for '), F.col('record_id')).alias('finding_summary'),
    F.to_json(F.struct('record_id', 'issue_type', 'duplicate_rank', 'ordering_key', 'ingest_time_utc')).alias('evidence_json'),
    F.lit('silver_stream_contract_issues').alias('source_table'),
)

schema_hits_base_df = contract_issues_df.filter(F.col('issue_type') != 'DUPLICATE_EVENT_ID').select(
    F.col('record_id').alias('evidence_key'),
    F.array(F.col('record_id')).alias('evidence_event_ids'),
    F.col('ordering_key').alias('affected_ordering_key'),
    F.coalesce(F.col('event_time_ts'), F.col('ingest_time_ts')).alias('finding_event_time_utc'),
    F.lit('schema_issue').alias('measure_name'),
    F.col('issue_type').alias('measured_value_text'),
    F.concat(F.lit('Replay contract validation failed for '), F.col('record_id')).alias('finding_summary'),
    F.to_json(F.struct('record_id', 'issue_type', 'event_type', 'source_system', 'source_instance_id')).alias('evidence_json'),
    F.lit('silver_stream_contract_issues').alias('source_table'),
)

late_hits_base_df = measures_df.filter(F.col('is_late_event') | F.col('timing_rule_exceeded')).select(
    F.col('event_id').alias('evidence_key'),
    F.array(F.col('event_id')).alias('evidence_event_ids'),
    F.col('ordering_key').alias('affected_ordering_key'),
    F.col('event_time_ts').alias('finding_event_time_utc'),
    F.lit('ingest_lag_seconds').alias('measure_name'),
    F.concat(F.lit('ingest_lag_seconds='), F.col('ingest_lag_seconds').cast('string'), F.lit('; command_processing_delay_ms='), F.coalesce(F.col('command_processing_delay_ms').cast('string'), F.lit('na'))).alias('measured_value_text'),
    F.concat(F.lit('Event exceeded the invented replay freshness budget for '), F.col('source_instance_id')).alias('finding_summary'),
    F.to_json(F.struct('event_id', 'event_type', 'source_system', 'source_instance_id', 'ingest_lag_seconds', 'command_processing_delay_ms')).alias('evidence_json'),
    F.lit('silver_streaming_measures').alias('source_table'),
)

dropout_hits_base_df = measures_df.filter((F.col('event_type') == 'sensor.observation') & (F.col('has_sequence_gap') | F.col('has_continuity_gap') | F.col('low_continuity_signal'))).select(
    F.col('event_id').alias('evidence_key'),
    F.array(F.col('event_id')).alias('evidence_event_ids'),
    F.col('ordering_key').alias('affected_ordering_key'),
    F.col('event_time_ts').alias('finding_event_time_utc'),
    F.lit('continuity_gap_seconds').alias('measure_name'),
    F.concat(F.lit('sequence_gap_count='), F.col('sequence_gap_count').cast('string'), F.lit('; continuity_gap_seconds='), F.col('continuity_gap_seconds').cast('string'), F.lit('; low_continuity='), F.col('low_continuity_signal').cast('string')).alias('measured_value_text'),
    F.concat(F.lit('Observation continuity degraded for ordering key '), F.col('ordering_key')).alias('finding_summary'),
    F.to_json(F.struct('event_id', 'ordering_key', 'sequence_gap_count', 'continuity_gap_seconds', 'quality_continuity_score')).alias('evidence_json'),
    F.lit('silver_streaming_measures').alias('source_table'),
)

missing_hits_base_df = expectations_df.filter(F.col('expectation_status').isin('MISSING_OR_OUT_OF_ORDER', 'OVERDUE', 'LATE')).select(
    F.col('expectation_id').alias('evidence_key'),
    F.col('evidence_event_ids'),
    F.concat_ws('|', F.coalesce(F.col('correlation_id'), F.lit('no-correlation')), F.coalesce(F.col('target_instance_id'), F.lit('no-target'))).alias('affected_ordering_key'),
    F.col('due_by_utc').alias('finding_event_time_utc'),
    F.lit('expectation_status').alias('measure_name'),
    F.col('expectation_status').alias('measured_value_text'),
    F.concat(F.lit('Expected '), F.col('expected_name'), F.lit(' after '), F.col('trigger_name'), F.lit(' is '), F.lower(F.col('expectation_status')), F.lit('.')).alias('finding_summary'),
    F.to_json(F.struct('expectation_id', 'expectation_type', 'trigger_name', 'expected_name', 'source_event_id', 'actual_event_id', 'due_by_utc', 'expectation_status')).alias('evidence_json'),
    F.lit('silver_stream_expectations').alias('source_table'),
)

duplicate_hits_df = apply_rule_metadata(duplicate_hits_base_df, 'duplicate_event', 'Duplicate event identifier observed in replay', 'Only one canonical record is retained per event_id')
schema_hits_df = apply_rule_metadata(schema_hits_base_df, 'schema_violation', 'Replay contract violation detected', 'Event identifiers, timestamps, classification, and site fields must remain valid in this demo')
late_hits_df = apply_rule_metadata(late_hits_base_df, 'late_event', 'Late or delayed stream event', 'Ingest lag must stay within 90 seconds and command delay within 90000 ms in this demo')
dropout_hits_df = apply_rule_metadata(dropout_hits_base_df, 'source_dropout', 'Source dropout signal detected', 'Observation continuity must avoid sequence gaps and continuity gaps over 120 seconds in this demo')
missing_hits_df = apply_rule_metadata(missing_hits_base_df, 'missing_expected_event', 'Missing expected follow-on event', 'An abstract assignment must be acknowledged within 180 seconds in this demo')

rule_hits_df = (
    duplicate_hits_df
    .unionByName(schema_hits_df, allowMissingColumns=True)
    .unionByName(late_hits_df, allowMissingColumns=True)
    .unionByName(dropout_hits_df, allowMissingColumns=True)
    .unionByName(missing_hits_df, allowMissingColumns=True)
    .select(
        'finding_id',
        'finding_version',
        'finding_status',
        'ruleset_id',
        'ruleset_version',
        'rule_id',
        'rule_version',
        'finding_code',
        'finding_category',
        'severity',
        'finding_title',
        'source_table',
        'affected_ordering_key',
        'measure_name',
        'measured_value_text',
        'threshold_text',
        'finding_summary',
        'evidence_key',
        'evidence_event_ids',
        'finding_event_time_utc',
        'evidence_json',
    )
)

rule_hits_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('silver_stream_rule_hits')
print(f'Rule hits ready: {rule_hits_df.count()}')
spark.table('silver_stream_rule_hits').orderBy('finding_code', 'evidence_key').show(truncate=False)
spark.table('silver_stream_rule_hits').groupBy('finding_code', 'severity').count().orderBy('finding_code').show(truncate=False)
